# Debugging Agents

**What you will build:** the habits for finding out *why* an agent misbehaved. Because agents are non-deterministic (0.0), you don't debug them by staring at code — you **inspect what actually happened**: the messages, the tool calls, the tokens. This is the code-course answer to n8n's "read the execution panel."

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/19_debugging_agents.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## The golden rule: read the transcript

When an agent does something wrong, the first question is always *what did the model actually see and do?* PydanticAI's `result.all_messages()` is the full record — the same message list you built by hand in Block 0. Let's debug a real failure with it.

Here's an agent whose tool the model **refuses to use**, because the description is useless:

In [ ]:
broken = Agent(model, system_prompt="You are a helpful assistant.")

@broken.tool_plain
def thing(x: str) -> str:
    """does stuff"""                      # <- useless description
    return {"AAPL": "224.10", "GOOGL": "178.30"}.get(x.upper(), "unknown")

result = await broken.run("What is the price of GOOGL?")
print("answer:", result.output)

The answer is probably wrong or made up. **Why?** Don't guess — inspect the transcript and see whether the tool was even called:

In [ ]:
def show_tool_calls(result):
    calls = [(p.tool_name, p.args) for m in result.all_messages()
             for p in m.parts if getattr(p, 'part_kind', '') == 'tool-call']
    return calls or "NO TOOLS CALLED — the model answered on its own"

print(show_tool_calls(result))

There it is: **no tool was called.** The model couldn't tell from `"does stuff"` that `thing` returns stock prices, so it skipped it and hallucinated. The transcript turned a mystery into a one-line diagnosis. Now fix the description and confirm the tool gets used:

In [ ]:
fixed = Agent(model, system_prompt="You are a helpful assistant.")

@fixed.tool_plain
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol like 'GOOGL'. Use this for any stock-price question."""
    return {"AAPL": "224.10", "GOOGL": "178.30"}.get(ticker.upper(), "unknown")

result = await fixed.run("What is the price of GOOGL?")
print("answer:", result.output)
print("tools called:", show_tool_calls(result))

Same failure, same fix as 1.2 — but now you found it by **evidence**, not intuition. That's the whole skill.

```{tip}
Nine out of ten agent bugs are one of: (1) a tool that wasn't called (bad description), (2) a tool that was called with bad arguments (ambiguous schema), or (3) a tool that returned something confusing. `all_messages()` shows you which, instantly.
```

## Watch the cost, too

A runaway agent shows up in the token count before you notice anything else. `result.usage()` totals the tokens across the whole run (all loop steps).

In [ ]:
u = result.usage()
print("tokens this run:", u.total_tokens)
print("(a sudden jump here usually means the agent is looping or re-reading a huge context)")

## Tracing: see every step, automatically

Printing `all_messages()` works, but for real systems you want a trace UI. **Logfire** (from the Pydantic team) records every model call, tool call, and token with one setup — the same tool you met in 1.6. It needs a free account, so this cell is illustrative; uncomment to use it.

In [ ]:
# !pip install -q "pydantic-ai-slim[logfire]"
# import logfire
# logfire.configure()               # asks for a free token on first run
# logfire.instrument_pydantic_ai()  # now EVERY agent.run(...) is traced with full detail
#
# await fixed.run("What is the price of AAPL?")   # open the Logfire link to see the trace
print("Uncomment to trace runs in Logfire. For the LangGraph side, LangSmith does the same job (below).")

```{note}
**LangGraph/LangChain** have the equivalent in **LangSmith**: set two environment variables and traces appear automatically — no code change.

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "..."   # from smith.langchain.com
```

Same idea as Logfire: stop guessing, look at the trace.
```

## A debugging checklist

| Symptom | First thing to check | Likely fix |
|---------|----------------------|-----------|
| Wrong / made-up answer | `all_messages()` — was a tool called? | Sharpen the tool description (1.2) |
| Tool called with bad args | The `tool-call` part's `args` | Add an example to the docstring |
| Runs forever / expensive | `usage().total_tokens` per run | Add a turn/recursion cap; check for loops |
| Ignores an instruction | The system prompt in the transcript | Tighten/de-conflict the rules (appendix) |
| Fails only sometimes | Run it 5x; it's non-determinism | Lower `temperature`; add an eval (1.6) |

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Diagnose a bad-args bug.** Give a tool an argument named `x` with no description, ask a question that needs it, and read the `tool-call` part's `args` to see what the model guessed.
2. **Catch a loop.** Build an agent whose tool always says "try again" and watch `usage().total_tokens` climb — then add a stopping rule.
3. **Trace it.** If you have a Logfire account, uncomment the tracing cell and compare the trace UI to the printed `all_messages()`.
::::

**Block 1 complete.** You can build, evaluate, and now debug clean typed agents — the toolkit for most real applications.

**What's next — Block 2:** when a task needs explicit **state, branching, cycles, or multiple agents**, you graduate to **LangGraph**. First stop, **2.0**: the same agent as a graph — and you finally get a picture to look at.